

Modelo EEG específico por narrativa. Los outer folds se leen desde data_partitions_paper_ready.csv; los sujetos de la partición que no estén en 04_only_eeg.csv se ignoran automáticamente mediante un merge interno.

Correr en COLAB

In [ ]:

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

from pathlib import Path
import warnings

import numpy as np
import pandas as pd

from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

SEED = 13
N_SPLITS = 5

DATA_DIR = Path("/content/drive/My Drive/Colab Notebooks/TFM/DataSet")
INPUT_CSV = DATA_DIR / "04_only_eeg.csv"
PARTITIONS_PATH = DATA_DIR / "data_partitions_paper_ready.csv"


In [ ]:

# FUNCIONES 


def normalize_keys(df):
    df = df.copy()
    df["subject_id"] = df["subject_id"].astype(str).str.strip().str.upper()
    df["avatar"] = df["avatar"].astype(str).str.strip()
    return df


def load_partitions():
    """Carga directamente las particiones del paper."""
    partitions = pd.read_csv(PARTITIONS_PATH)
    return partitions[["subject_id", "avatar", "outer_fold"]].copy()


def make_sample_weight(y):
    """Pesos inversos a la frecuencia de clase dentro del train fold."""
    y = pd.Series(y)
    counts = y.value_counts()
    return y.map({cls: len(y) / (len(counts) * n) for cls, n in counts.items()}).to_numpy()


def make_model():
    """Pipeline EEG: escalado dentro del fold + XGBoost."""
    return Pipeline([
        ("scaler", StandardScaler()),
        ("xgb", XGBClassifier(
            objective="binary:logistic",
            eval_metric="auc",
            random_state=SEED,
            n_jobs=-1,
        )),
    ])


def safe_auc(y_true, y_prob):
    try:
        return roc_auc_score(y_true, y_prob)
    except ValueError:
        return np.nan


def compute_metrics(y_true, y_pred, y_prob):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "auc": safe_auc(y_true, y_prob),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall_pos": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "recall_neg": recall_score(y_true, y_pred, pos_label=0, zero_division=0),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }


def summarize_metrics(metrics_df):
    cols = ["accuracy", "balanced_accuracy", "f1", "auc", "precision", "recall_pos", "recall_neg", "kappa"]
    return pd.concat([
        metrics_df[cols].mean().round(3).rename("mean"),
        metrics_df[cols].std().round(3).rename("std"),
    ], axis=1)


In [ ]:

# CARGAR EEG 


df = normalize_keys(pd.read_csv(INPUT_CSV))
df = df.dropna(subset=["label"]).copy()
df["label"] = df["label"].astype(int)

partitions = load_partitions()

# Inner merge: conserva únicamente las filas EEG que aparecen en data_partitions_paper_ready.
# Si hay sujetos en data_partitions que no están en 04_only_eeg.csv, simplemente se quedan fuera.
df_keys = set(map(tuple, df[["subject_id", "avatar"]].drop_duplicates().to_numpy()))
part_keys = set(map(tuple, partitions[["subject_id", "avatar"]].drop_duplicates().to_numpy()))
missing_in_eeg = sorted({subject for subject, avatar in (part_keys - df_keys)})
extra_in_eeg = sorted({subject for subject, avatar in (df_keys - part_keys)})

df = df.merge(partitions, on=["subject_id", "avatar"], how="inner")

meta_cols = {"subject_id", "avatar", "label", "phq", "outer_fold", "outer_repeat", "narrative"}
feature_cols = [
    c for c in df.columns
    if c not in meta_cols and pd.api.types.is_numeric_dtype(df[c])
]

missing_total = int(df[feature_cols].isna().sum().sum())
if missing_total > 0:
    raise ValueError("Hay valores faltantes en EEG. Revisa 04_only_eeg.csv antes de entrenar.")

print("Filas EEG con partición:", len(df))
print("Sujetos EEG con partición:", df["subject_id"].nunique())
print("Variables EEG:", len(feature_cols))
print("Sujetos en data_partitions que no aparecen en EEG:", len(missing_in_eeg))
print(missing_in_eeg[:20])
print("Sujetos en EEG que no aparecen en data_partitions:", len(extra_in_eeg))
print(extra_in_eeg[:20])

print("\nSujetos por outer fold:")
display(df.groupby("outer_fold")["subject_id"].nunique().to_frame("n_subjects"))

print("\nDistribución de clases por outer fold:")
subjects_fold = df[["subject_id", "label", "outer_fold"]].drop_duplicates("subject_id")
display(pd.crosstab(subjects_fold["outer_fold"], subjects_fold["label"]))


Filas EEG con partición: 558
Sujetos EEG con partición: 94
Variables EEG: 27
Sujetos en data_partitions que no aparecen en EEG: 7
['USER_17_CB', 'USER_18_CB', 'USER_19_CB', 'USER_20_CB', 'USER_46_CB2', 'USER_47_CB2', 'USER_48_CB2']
Sujetos en EEG que no aparecen en data_partitions: 0
[]

Sujetos por outer fold:


,n_subjects
outer_fold,
1,20
2,18
3,18
4,19
5,19



Distribución de clases por outer fold:


label,0,1
outer_fold,,
1,11,9
2,10,8
3,11,7
4,12,7
5,11,8


In [ ]:


narratives = ["Neutral1", "Neutral2", "Happy", "Sad", "Angry", "Relax"]
df["narrative"] = df["avatar"].astype(str)
df = df[df["narrative"].isin(narratives)].copy()

print("Narrativas disponibles:")
display(df["narrative"].value_counts().reindex(narratives).to_frame("n_rows"))


Narrativas disponibles:


,n_rows
narrative,
Neutral1,94
Neutral2,92
Happy,94
Sad,94
Angry,92
Relax,92


In [ ]:

# MODELO + HIPERPARÁMETROS


PARAM_GRID = {
    "xgb__max_depth": list(range(3, 12)),
    "xgb__learning_rate": [0.001, 0.01, 0.1, 1.0],
    "xgb__n_estimators": [50, 100],
}

SCORING = {
    "UAcc": "balanced_accuracy",
    "auc": "roc_auc",
    "f1": "f1",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
}


In [ ]:

# NESTED CV 


OUT_DIR = DATA_DIR / "02_results_eeg_narrative_paper_folds_unimodal"
OUT_DIR.mkdir(parents=True, exist_ok=True)

narrative_predictions = []
subject_predictions = []
subject_metrics = []
best_params_rows = []

for fold in sorted(df["outer_fold"].unique()):
    print(f"\n===== OUTER FOLD {fold} =====")
    fold_predictions = []

    for narrative in narratives:
        train_n = df[(df["outer_fold"] != fold) & (df["narrative"] == narrative)].copy()
        test_n = df[(df["outer_fold"] == fold) & (df["narrative"] == narrative)].copy()

        if train_n.empty or test_n.empty or train_n["label"].nunique() < 2:
            print(f"{narrative}: saltada")
            continue

        X_train = train_n[feature_cols]
        y_train = train_n["label"]
        groups_train = train_n["subject_id"]
        X_test = test_n[feature_cols]

        inner_cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

        grid = GridSearchCV(
            estimator=make_model(),
            param_grid=PARAM_GRID,
            scoring=SCORING,
            refit="UAcc",
            cv=inner_cv,
            n_jobs=-1,
            verbose=0,
        )
        grid.fit(X_train, y_train, groups=groups_train, xgb__sample_weight=make_sample_weight(y_train))

        prob = grid.best_estimator_.predict_proba(X_test)[:, 1]

        pred_n = test_n[["subject_id", "label", "narrative", "outer_fold"]].copy()
        pred_n["y_prob"] = prob
        fold_predictions.append(pred_n)
        narrative_predictions.append(pred_n)

        best_idx = grid.best_index_
        best_params_rows.append({
            "outer_fold": fold,
            "narrative": narrative,
            "best_inner_UAcc": grid.cv_results_["mean_test_UAcc"][best_idx],
            "best_inner_auc": grid.cv_results_["mean_test_auc"][best_idx],
            "best_inner_f1": grid.cv_results_["mean_test_f1"][best_idx],
            "best_max_depth": grid.best_params_["xgb__max_depth"],
            "best_learning_rate": grid.best_params_["xgb__learning_rate"],
            "best_n_estimators": grid.best_params_["xgb__n_estimators"],
        })

        print(f"{narrative}: best inner UAcc={grid.cv_results_['mean_test_UAcc'][best_idx]:.3f}")

    fold_pred = pd.concat(fold_predictions, ignore_index=True)
    subj_pred = fold_pred.groupby(["subject_id", "label", "outer_fold"], as_index=False)["y_prob"].mean()
    subj_pred["y_pred"] = (subj_pred["y_prob"] >= 0.5).astype(int)
    subject_predictions.append(subj_pred)

    subj_m = compute_metrics(subj_pred["label"], subj_pred["y_pred"], subj_pred["y_prob"])
    subj_m["outer_fold"] = fold
    subject_metrics.append(subj_m)

    print("Subject F1:", round(subj_m["f1"], 3))

narrative_predictions_df = pd.concat(narrative_predictions, ignore_index=True)
subject_predictions_df = pd.concat(subject_predictions, ignore_index=True)
subject_metrics_df = pd.DataFrame(subject_metrics)
best_params_df = pd.DataFrame(best_params_rows)

narrative_predictions_df.to_csv(OUT_DIR / "eeg_probabilities_by_narrative.csv", index=False)
subject_predictions_df.to_csv(OUT_DIR / "eeg_subject_predictions.csv", index=False)
subject_metrics_df.to_csv(OUT_DIR / "eeg_subject_outer_metrics.csv", index=False)
best_params_df.to_csv(OUT_DIR / "eeg_best_params_by_fold_and_narrative.csv", index=False)

print("\nSubject-level metrics")
display(summarize_metrics(subject_metrics_df))

print("\nArchivos guardados en:", OUT_DIR)



===== OUTER FOLD 1 =====
Neutral1: best inner UAcc=0.662
Neutral2: best inner UAcc=0.741
Happy: best inner UAcc=0.691
Sad: best inner UAcc=0.672
Angry: best inner UAcc=0.566
Relax: best inner UAcc=0.693
Subject F1: 0.706

===== OUTER FOLD 2 =====
Neutral1: best inner UAcc=0.710
Neutral2: best inner UAcc=0.737
Happy: best inner UAcc=0.687
Sad: best inner UAcc=0.711
Angry: best inner UAcc=0.650
Relax: best inner UAcc=0.749
Subject F1: 0.667

===== OUTER FOLD 3 =====
Neutral1: best inner UAcc=0.673
Neutral2: best inner UAcc=0.698
Happy: best inner UAcc=0.736
Sad: best inner UAcc=0.628
Angry: best inner UAcc=0.577
Relax: best inner UAcc=0.636
Subject F1: 0.706

===== OUTER FOLD 4 =====
Neutral1: best inner UAcc=0.611
Neutral2: best inner UAcc=0.721
Happy: best inner UAcc=0.736
Sad: best inner UAcc=0.667
Angry: best inner UAcc=0.562
Relax: best inner UAcc=0.571
Subject F1: 0.923

===== OUTER FOLD 5 =====
Neutral1: best inner UAcc=0.801
Neutral2: best inner UAcc=0.781
Happy: best inner UAcc

,mean,std
accuracy,0.754,0.113
balanced_accuracy,0.744,0.115
f1,0.680,0.186
auc,0.796,0.102
precision,0.790,0.201
recall_pos,0.676,0.251
recall_neg,0.811,0.191
kappa,0.490,0.236



Archivos guardados en: /content/drive/My Drive/Colab Notebooks/TFM/DataSet/02_results_eeg_narrative_paper_folds_unimodal
